# ControlledRAG optional modern judge — build-only notebook

**Default status: PREPARED_NOT_EXECUTED.** This notebook performs only the synthetic smoke test unless both the notebook flag and the copied run-config flag are explicitly changed to `True`. A real run is fixed-output rescoring, not retrieval or generation.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, shutil, subprocess, sys

RUN_REAL_INFERENCE = False
PACKAGE_DIR = Path.cwd()
CONFIG_PATH = PACKAGE_DIR / 'run_config.json'
print({'RUN_REAL_INFERENCE': RUN_REAL_INFERENCE, 'package_dir': str(PACKAGE_DIR)})

In [ ]:
# CPU-only default path. It performs no network call and loads no model.
if not RUN_REAL_INFERENCE:
    completed = subprocess.run(
        [sys.executable, str(PACKAGE_DIR / 'CPU_SYNTHETIC_SMOKE_TEST.py')],
        cwd=PACKAGE_DIR,
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout.strip())
    print('Synthetic smoke test complete. Real execution remains disabled.')

In [ ]:
# Explicit real-run gate and hard-pin validation.
if RUN_REAL_INFERENCE:
    config = json.loads(CONFIG_PATH.read_text())
    assert config['run_real_inference'] is True
    forbidden = {'auto', 'PIN_PROVIDER_BEFORE_EXECUTION', 'PIN_EXACT_MODEL_ID_OR_REVISION'}
    assert config['provider'] not in forbidden
    assert config['model'] not in forbidden
    assert config['model'] in config['allowed_returned_models']
    assert config['allow_fallback_attempts'] == 0
    assert config['quantization'] in {'none', '4bit', '8bit'}
    assert config['dtype'] in {'bf16', 'fp16', 'fp32'}
    assert config['gpu_strategy'] in {'one_worker_per_gpu', 'device_map_auto'}
else:
    config = None

In [ ]:
# T4x2 detection and strategy gate. torch is imported only for an authorized run.
if RUN_REAL_INFERENCE:
    import torch
    gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
    gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
    t4x2_detected = gpu_count == 2 and all('T4' in name for name in gpu_names)
    runtime_metadata = {
        'gpu_count': gpu_count,
        'gpu_names': gpu_names,
        't4x2_detected': t4x2_detected,
        'strategy': config['gpu_strategy'],
        'quantization': config['quantization'],
        'dtype': config['dtype'],
    }
    assert gpu_count >= 1, 'Real Kaggle run requires a CUDA GPU'
    if config['gpu_strategy'] == 'one_worker_per_gpu':
        assert gpu_count == 2, 'one_worker_per_gpu expects the requested T4x2 session'
    print(runtime_metadata)

In [ ]:
# Load and verify the fixed source plus text-free candidate; reject duplicates.
if RUN_REAL_INFERENCE:
    from SAMPLE_SELECTION import EXPECTED_SOURCE_SHA256, canonical_digest, sha256_file
    source_path = Path(config['source_csv'])
    manifest_path = PACKAGE_DIR / config['candidate_manifest']
    assert config['source_sha256'] == EXPECTED_SOURCE_SHA256
    assert sha256_file(source_path) == EXPECTED_SOURCE_SHA256
    candidate_index = json.loads((PACKAGE_DIR / 'SAMPLE_MANIFEST_CANDIDATES' / 'candidates_index.json').read_text())
    expected_manifest_hashes = {record['file']: record['sha256'] for record in candidate_index['candidate_records']}
    assert manifest_path.name in expected_manifest_hashes
    assert sha256_file(manifest_path) == expected_manifest_hashes[manifest_path.name]
    with source_path.open(newline='', encoding='utf-8') as handle:
        source_rows = list(csv.DictReader(handle))
    with manifest_path.open(newline='', encoding='utf-8') as handle:
        manifest_rows = list(csv.DictReader(handle))
    assert len({row['row_id'] for row in manifest_rows}) == len(manifest_rows)
    result_dir = PACKAGE_DIR / Path(config['output_jsonl']).parent
    result_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CONFIG_PATH, result_dir / 'run_config.json')
    shutil.copy2(manifest_path, result_dir / 'candidate_manifest.csv')
    joined_rows = []
    for manifest_row in manifest_rows:
        source = source_rows[int(manifest_row['source_row_index'])]
        assert canonical_digest(source, ('question','ground_truth','condition','answer','context','dataset','seed','model')) == manifest_row['input_digest']
        joined_rows.append({**manifest_row, 'question': source['question'], 'context': source['context'], 'answer': source['answer']})
    print({'candidate_rows': len(joined_rows), 'unique_rows': len({r['row_id'] for r in joined_rows})})

In [ ]:
# Checkpoint/resume contract and prompt construction.
if RUN_REAL_INFERENCE:
    from provider_adapter import JudgeRequest, RoutingPolicy, make_result_record, parse_strict_output, utc_now
    prompt_template = (PACKAGE_DIR / 'PROMPT_TEMPLATE.md').read_text()
    result_path = PACKAGE_DIR / config['output_jsonl']
    attempt_path = result_path.with_name('attempt_log.jsonl')
    result_path.parent.mkdir(parents=True, exist_ok=True)
    scratch_dir = result_path.parent / 'scratch'
    scratch_dir.mkdir(parents=True, exist_ok=True)
    existing_records = {}
    resume_paths = [result_path, *sorted(scratch_dir.glob('shard_*_results.jsonl'))]
    if config['resume']:
        for resume_path in resume_paths:
            if not resume_path.exists(): continue
            for line in resume_path.read_text().splitlines():
                if not line.strip(): continue
                record = json.loads(line)
                if record['row_id'] in existing_records and existing_records[record['row_id']] != record:
                    raise ValueError('conflicting duplicate row during resume')
                existing_records[record['row_id']] = record
    completed_ids = set(existing_records)
    pending_rows = [row for row in joined_rows if row['row_id'] not in completed_ids]
    def render_prompt(row):
        rendered = prompt_template
        for placeholder, value in (
            ('{{question}}', row['question']),
            ('{{context}}', row['context']),
            ('{{answer}}', row['answer']),
        ):
            if placeholder not in rendered:
                raise ValueError(f'missing frozen prompt placeholder: {placeholder}')
            rendered = rendered.replace(placeholder, value)
        if '{{' in rendered or '}}' in rendered:
            raise ValueError('unresolved placeholder in frozen prompt')
        return rendered
    print({'completed': len(completed_ids), 'pending': len(pending_rows)})

In [ ]:
# Real execution: one worker per GPU for fitting models, or one device-map worker for larger models.
if RUN_REAL_INFERENCE:
    from concurrent.futures import ThreadPoolExecutor
    def run_hf_shard(rows, device_index):
        from huggingface_adapter import HuggingFaceAdapter
        model_id, revision = config['model'].rsplit('@', 1)
        policy = RoutingPolicy(config['provider'], config['model'], tuple(config['allowed_returned_models']), tuple(config['allowed_routed_via']), config['allow_fallback_attempts'])
        adapter = HuggingFaceAdapter(policy=policy, model_id=model_id, revision=revision, quantization=config['quantization'], dtype=config['dtype'], gpu_strategy=config['gpu_strategy'], device_index=device_index, local_files_only=config['local_files_only'], allow_model_load=True)
        final_records, attempt_records = [], []
        shard_result_path = scratch_dir / f'shard_{device_index}_results.jsonl'
        shard_attempt_path = scratch_dir / f'shard_{device_index}_attempts.jsonl'
        shard_checkpoint_path = scratch_dir / f'shard_{device_index}_checkpoint.json'
        for row_index, row in enumerate(rows, start=1):
            request = JudgeRequest(config['run_id'], row['row_id'], row['input_digest'], render_prompt(row))
            started = utc_now()
            try:
                response = adapter.execute(request)
                parsed = parse_strict_output(response.raw_output)
                record = make_result_record(request=request, policy=policy, started_at_utc=started, finished_at_utc=utc_now(), status='ok', response=response, parsed_output=parsed)
            except Exception as exc:
                record = make_result_record(request=request, policy=policy, started_at_utc=started, finished_at_utc=utc_now(), status='provider_error', error=f'{type(exc).__name__}: {exc}')
            attempt_records.append(record)
            final_records.append(record)
            with shard_attempt_path.open('a', encoding='utf-8') as handle: handle.write(json.dumps(record, sort_keys=True) + '\n')
            with shard_result_path.open('a', encoding='utf-8') as handle: handle.write(json.dumps(record, sort_keys=True) + '\n')
            if row_index % config['checkpoint_every'] == 0 or row_index == len(rows):
                shard_checkpoint_path.write_text(json.dumps({'device_index': device_index, 'completed_row_ids': [r['row_id'] for r in final_records]}, indent=2) + '\n')
        return final_records, attempt_records
    if config['gpu_strategy'] == 'one_worker_per_gpu':
        shards = [pending_rows[i::gpu_count] for i in range(gpu_count)]
        with ThreadPoolExecutor(max_workers=gpu_count) as pool:
            shard_outputs = list(pool.map(run_hf_shard, shards, range(gpu_count)))
    else:
        shard_outputs = [run_hf_shard(pending_rows, 0)]
    final_records = [record for finals, _ in shard_outputs for record in finals]
    merged_records = dict(existing_records)
    for record in final_records:
        if record['row_id'] in merged_records: raise ValueError('duplicate final row')
        merged_records[record['row_id']] = record
    assert set(merged_records).issubset({row['row_id'] for row in manifest_rows})
    with result_path.open('w', encoding='utf-8') as handle:
        for row in manifest_rows:
            if row['row_id'] in merged_records: handle.write(json.dumps(merged_records[row['row_id']], sort_keys=True) + '\n')
    attempt_lines = []
    for path in sorted(scratch_dir.glob('shard_*_attempts.jsonl')):
        attempt_lines.extend(line for line in path.read_text().splitlines() if line.strip())
    attempt_path.write_text('\n'.join(attempt_lines) + ('\n' if attempt_lines else ''), encoding='utf-8')
    checkpoint = {'run_id': config['run_id'], 'completed_row_ids': sorted(merged_records)}
    (PACKAGE_DIR / config['checkpoint_json']).write_text(json.dumps(checkpoint, indent=2) + '\n')
    (result_path.parent / 'runtime_metadata.json').write_text(json.dumps(runtime_metadata, indent=2) + '\n')
    print({'new_results': len(final_records), 'result_path': str(result_path)})

In [ ]:
# Package only after post-run validation. Both flags are false by default.
PACKAGE_RESULT_ZIP = False
if RUN_REAL_INFERENCE and PACKAGE_RESULT_ZIP:
    analysis_path = result_path.parent / 'post_run_analysis.json'
    analysis = json.loads(analysis_path.read_text())
    assert analysis['scientifically_valid'] is True
    subprocess.run([sys.executable, str(PACKAGE_DIR / 'BUILD_RESULT_ZIP.py'), '--input-dir', str(result_path.parent), '--output-zip', str(PACKAGE_DIR / 'controlledrag_modern_judge_results.zip')], check=True)
elif RUN_REAL_INFERENCE:
    print('Run POST_RUN_ANALYSIS.py, inspect scientifically_valid, then set PACKAGE_RESULT_ZIP=True. The optional run is not required rebuttal evidence.')